# Machine Learning

In [56]:
import os
os.environ["SCIPY_ARRAY_API"] = "1" 

In [57]:
import sklearn
sklearn.set_config(array_api_dispatch=True)

In [ ]:
import cupy as cp
import pandas as pd
import numpy as np
from sklearn.model_selection import (
    train_test_split,
    PredefinedSplit,
    RandomizedSearchCV
)
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_squared_error, 
    r2_score
)
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
import xgboost as xgb
from scipy.stats import (
    uniform, 
    randint,
    loguniform
)
import matplotlib.pyplot as plt
import copy
from pprint import pprint
import json
from statsmodels.stats.outliers_influence import variance_inflation_factor
import torch
from xgboost import XGBRegressor

In [59]:
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [60]:
initial_df = pd.read_parquet("../data/processed/anime_data_2.parquet")
initial_df.info()

<class 'pandas.DataFrame'>
Index: 5350 entries, 0 to 71
Data columns (total 87 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   mal_id                  5350 non-null   int64  
 1   title                   5350 non-null   str    
 2   producers               5350 non-null   object 
 3   genres                  5350 non-null   object 
 4   studios                 5350 non-null   object 
 5   demographics            5350 non-null   object 
 6   themes                  5350 non-null   object 
 7   rating                  5309 non-null   str    
 8   sequel                  5350 non-null   bool   
 9   members                 5350 non-null   int64  
 10  thumbnail               5350 non-null   bool   
 11  prequel_score           1388 non-null   float64
 12  prequel_members         1402 non-null   float64
 13  prequel_type            1402 non-null   str    
 14  cohort                  5350 non-null   str    
 15  drop_

In [61]:
df = initial_df.head(5278)
real_df = initial_df.tail(72)

print(df.info())
print(real_df.info())

<class 'pandas.DataFrame'>
Index: 5278 entries, 0 to 5307
Data columns (total 87 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   mal_id                  5278 non-null   int64  
 1   title                   5278 non-null   str    
 2   producers               5278 non-null   object 
 3   genres                  5278 non-null   object 
 4   studios                 5278 non-null   object 
 5   demographics            5278 non-null   object 
 6   themes                  5278 non-null   object 
 7   rating                  5278 non-null   str    
 8   sequel                  5278 non-null   bool   
 9   members                 5278 non-null   int64  
 10  thumbnail               5278 non-null   bool   
 11  prequel_score           1364 non-null   float64
 12  prequel_members         1376 non-null   float64
 13  prequel_type            1376 non-null   str    
 14  cohort                  5278 non-null   str    
 15  dro

## Input Preparation

Plan:
* adaptation_members and prequel_members obey a power law, which could be problematic
* fill NaN values with zeroes and let the model learn from the bools (this imputation will be handled in notebook 1 soon)

In [62]:
df = df.reset_index(drop=True)

In [63]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5278 entries, 0 to 5277
Data columns (total 87 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   mal_id                  5278 non-null   int64  
 1   title                   5278 non-null   str    
 2   producers               5278 non-null   object 
 3   genres                  5278 non-null   object 
 4   studios                 5278 non-null   object 
 5   demographics            5278 non-null   object 
 6   themes                  5278 non-null   object 
 7   rating                  5278 non-null   str    
 8   sequel                  5278 non-null   bool   
 9   members                 5278 non-null   int64  
 10  thumbnail               5278 non-null   bool   
 11  prequel_score           1364 non-null   float64
 12  prequel_members         1376 non-null   float64
 13  prequel_type            1376 non-null   str    
 14  cohort                  5278 non-null   str    
 15

In [64]:
df = df.fillna({'prequel_score': 0, 'prequel_members': 0, 'prequel_type': "", 'adaptation_score': 0,'adaptation_members': 0})
real_df = real_df.fillna({'prequel_score': 0, 'prequel_members': 0, 'prequel_type': "", 'adaptation_score': 0,'adaptation_members': 0})

cols_to_log = ['prequel_members'] # this will change in the third data pass!
df[cols_to_log] = np.log1p(df[cols_to_log])
real_df[cols_to_log] = np.log1p(real_df[cols_to_log])
print(df[cols_to_log].describe())
print(real_df[cols_to_log].describe())

       prequel_members
count      5278.000000
mean          2.800895
std           4.852111
min           0.000000
25%           0.000000
50%           0.000000
75%           6.769399
max          15.300479
       prequel_members
count        72.000000
mean          3.993404
std           5.503425
min           0.000000
25%           0.000000
50%           0.000000
75%          10.334042
max          14.474516


Let's test for alignment:

In [65]:
print(df.info())
print(real_df.info())

<class 'pandas.DataFrame'>
RangeIndex: 5278 entries, 0 to 5277
Data columns (total 87 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   mal_id                  5278 non-null   int64  
 1   title                   5278 non-null   str    
 2   producers               5278 non-null   object 
 3   genres                  5278 non-null   object 
 4   studios                 5278 non-null   object 
 5   demographics            5278 non-null   object 
 6   themes                  5278 non-null   object 
 7   rating                  5278 non-null   str    
 8   sequel                  5278 non-null   bool   
 9   members                 5278 non-null   int64  
 10  thumbnail               5278 non-null   bool   
 11  prequel_score           5278 non-null   float64
 12  prequel_members         5278 non-null   float64
 13  prequel_type            5278 non-null   str    
 14  cohort                  5278 non-null   str    
 15

In [66]:
X_other_pre = df.drop(columns=['thumbnail', 'members', 'title', 'mal_id', 'cohort', 'producers', 'genres', 'studios', 'demographics', 'themes', 'score_z', 'wc_z', 'favorites_z', 'dropped_z', 'drop_rate', 'drop_rate_z', 'forum_z'])
X_other_pre = X_other_pre.reset_index(drop=True)
X_other_pre = pd.get_dummies(X_other_pre, columns=['rating'], dtype=int, drop_first=True)
X_other_pre = pd.get_dummies(X_other_pre, columns=['prequel_type'], dtype=int, drop_first=True)
X_other_pre['sequel'] = X_other_pre['sequel'].astype(int)
X_other_pre.columns = X_other_pre.columns.str.replace(' ', '_')
print(X_other_pre.info())

real_other_pre = real_df.drop(columns=['thumbnail', 'members', 'title',  'mal_id', 'cohort', 'producers', 'genres', 'studios', 'demographics', 'themes', 'score_z', 'wc_z', 'favorites_z', 'dropped_z', 'drop_rate', 'drop_rate_z','forum_z'])
real_other_pre = real_other_pre.reset_index(drop=True)
real_other_pre = pd.get_dummies(real_other_pre, columns=['rating'], dtype=int)
real_other_pre = pd.get_dummies(real_other_pre, columns=['prequel_type'], dtype=int)
real_other_pre['sequel'] = real_other_pre['sequel'].astype(int)
real_other_pre.columns = real_other_pre.columns.str.replace(' ', '_')
print(real_other_pre.info())

<class 'pandas.DataFrame'>
RangeIndex: 5278 entries, 0 to 5277
Data columns (total 80 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   sequel                                 5278 non-null   int64  
 1   prequel_score                          5278 non-null   float64
 2   prequel_members                        5278 non-null   float64
 3   adaptation_score                       5278 non-null   float64
 4   adaptation_members                     5278 non-null   float64
 5   has_adaptation_score                   5278 non-null   int64  
 6   has_adaptation_members                 5278 non-null   int64  
 7   has_prequel_score                      5278 non-null   int64  
 8   has_prequel_members                    5278 non-null   int64  
 9   has_prequel_type                       5278 non-null   int64  
 10  genre_Action                           5278 non-null   int64  
 11  genre_Adventure

### Multicollinearity Test

In [67]:
def safe_boolean_vif(df):
    zero_variance_cols = df.columns[df.nunique() <= 1].tolist()
    if zero_variance_cols:
        print(f"Dropped zero-variance columns immediately: {zero_variance_cols}")
        df = df.drop(columns=zero_variance_cols)
        
    X = df.copy()
    X['constant'] = 1.0
    
    vif_data = pd.DataFrame()
    vif_data["feature"] = X.columns
    
    vifs = []
    for i in range(len(X.columns)):
        try:
            val = variance_inflation_factor(X.values, i)
            vifs.append(np.inf if np.isinf(val) or np.isnan(val) else val)
        except ZeroDivisionError:
            vifs.append(np.inf)
            
    vif_data["VIF"] = vifs
    
    return vif_data[vif_data["feature"] != 'constant'].reset_index(drop=True)

In [68]:
vif_results = safe_boolean_vif(X_other_pre)
print("\nFinal VIF Data:")
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(vif_results) 

c:\Users\14793\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\statsmodels\stats\outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)



Final VIF Data:


,feature,VIF
0,sequel,inf
1,prequel_score,124.938246
2,prequel_members,46.686421
3,adaptation_score,2.990360
4,adaptation_members,6.216886
5,has_adaptation_score,4.525568
6,has_adaptation_members,2.593336
7,has_prequel_score,173.857083
8,has_prequel_members,inf
9,has_prequel_type,inf


We can infer the following:
1. We have to remove the "has_prequel" variables. The main worry is that the neural network won't be able to learn from the imputed zeroes. However, since a zero is extremely rare or impossible for prequel score and prequel members, this should not be a big issue.
2. Prequel types will ruin our feature importance reliability. We can turn this into a "prequel_type_tv" bool instead since the only prequel type in the inference data is TV.

In [69]:
X_other_pre = X_other_pre.drop(columns=['has_prequel_score', 'has_prequel_members', 'has_prequel_type', 'prequel_type_Movie', 'prequel_type_Music',
                                        'prequel_type_ONA', 'prequel_type_OVA', 'prequel_type_PV', 'prequel_type_Special', 'prequel_type_TV', 'prequel_type_TV_Special'])

real_other_pre = real_other_pre.drop(columns=['has_prequel_score', 'has_prequel_members', 'has_prequel_type', 'prequel_type_'])

X_other_pre["prequel_type_tv"] = (df['prequel_type'] == "TV").astype(int)

X_other_pre["prequel_type_tv"].describe()

count    5278.000000
mean        0.214475
std         0.410497
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         1.000000
Name: prequel_type_tv, dtype: float64

Now lets run VIF one more time.

In [70]:
vif_results = safe_boolean_vif(X_other_pre)
print("\nFinal VIF Data:")
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(vif_results) 


Final VIF Data:


,feature,VIF
0,sequel,42.016975
1,prequel_score,64.468171
2,prequel_members,40.722750
3,adaptation_score,2.940022
4,adaptation_members,6.201957
5,has_adaptation_score,4.517740
6,has_adaptation_members,2.589171
7,genre_Action,1.725098
8,genre_Adventure,1.364706
9,genre_Comedy,1.628285


In [71]:
real_other_pre = real_other_pre.reindex(columns=X_other_pre.columns, fill_value=0)

print(X_other_pre.info())
print(real_other_pre.info())

<class 'pandas.DataFrame'>
RangeIndex: 5278 entries, 0 to 5277
Data columns (total 70 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   sequel                                 5278 non-null   int64  
 1   prequel_score                          5278 non-null   float64
 2   prequel_members                        5278 non-null   float64
 3   adaptation_score                       5278 non-null   float64
 4   adaptation_members                     5278 non-null   float64
 5   has_adaptation_score                   5278 non-null   int64  
 6   has_adaptation_members                 5278 non-null   int64  
 7   genre_Action                           5278 non-null   int64  
 8   genre_Adventure                        5278 non-null   int64  
 9   genre_Comedy                           5278 non-null   int64  
 10  genre_Drama                            5278 non-null   int64  
 11  genre_Ecchi    

In [72]:
assert list(X_other_pre.columns) == list(real_other_pre.columns), \
    set(X_other_pre.columns) ^ set(real_other_pre.columns)

## Baseline Model (Random Forest)

In [25]:
cols = ['score_z', 'wc_z', 'favorites_z', 'dropped_z', 'forum_z']

In [21]:
def multi_rf(X, y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    param_distributions = {
        'n_estimators':[150, 250, 400],
        'max_features': ['sqrt', 0.2, 0.3, 0.4],            
        'max_depth': [10, 15, 20, 25, None],                 
        'min_samples_split':[4, 6, 8],                  
        'min_samples_leaf': [2, 3, 4, 6]                     
    }

    rf_random_search = RandomizedSearchCV(
        estimator=RandomForestRegressor(random_state=42),
        param_distributions=param_distributions,
        n_iter=30,
        cv=5,
        scoring='neg_mean_squared_error',
        verbose=3,
        random_state=42,
        n_jobs=-1
    )

    print("Starting hyperparameter tuning...")
    rf_random_search.fit(X_train, y_train)

    print("\n--- Tuning Complete ---")
    print("Best Hyperparameters Found:", rf_random_search.best_params_)
    best_rf_model = rf_random_search.best_estimator_
    y_pred = best_rf_model.predict(X_test)

    mse_per_target = mean_squared_error(y_test, y_pred, multioutput='raw_values')
    r2_per_target = r2_score(y_test, y_pred, multioutput='raw_values')

    print("\n--- Test Set Evaluation Per Target ---")
    for i in range(y_test.shape[1]):
        print(f"Target {i+1} -> MSE: {mse_per_target[i]:.4f} | R² Score: {r2_per_target[i]:.4f}")

In [22]:
multi_rf(X_other_pre, df[cols])

Starting hyperparameter tuning...
Fitting 5 folds for each of 30 candidates, totalling 150 fits

--- Tuning Complete ---
Best Hyperparameters Found: {'n_estimators': 250, 'min_samples_split': 6, 'min_samples_leaf': 2, 'max_features': 0.2, 'max_depth': 20}

--- Test Set Evaluation Per Target ---
Target 1 -> MSE: 0.4429 | R² Score: 0.5262
Target 2 -> MSE: 0.3282 | R² Score: 0.6355
Target 3 -> MSE: 0.3434 | R² Score: 0.6229
Target 4 -> MSE: 0.3673 | R² Score: 0.5991
Target 5 -> MSE: 0.4502 | R² Score: 0.5147


## Improved Model (XGBoost)

In [73]:
def xgboost(X, y, col):
    X_full_train, X_test, y_full_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    X_train, X_val, y_train, y_val = train_test_split(X_full_train, y_full_train, test_size=0.1, random_state=42)

    split_index = [-1] * len(X_train) + [0] * len(X_val)
    pds = PredefinedSplit(test_fold=split_index)

    X_search_g = cp.asarray(np.vstack((X_train, X_val)))
    y_search_g = cp.asarray(np.concatenate((y_train, y_val)))

    X_train_g = cp.asarray(X_train)
    y_train_g = cp.asarray(y_train)
    X_val_g = cp.asarray(X_val)
    y_val_g = cp.asarray(y_val)
    X_test_g = cp.asarray(X_test)

    param_distributions = {
    'n_estimators': [500, 1000, 1500, 2000, 3000],
    'learning_rate': [0.01, 0.02, 0.05, 0.1],
    'max_depth': [2, 3, 4, 5, 6, 8],
    'min_child_weight': [1, 3, 5, 7, 10],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.5, 0.7, 0.8, 0.9, 1.0],
    'gamma': [0, 0.01, 0.1, 0.5, 1],
    'reg_alpha': [0, 0.01, 0.1, 1, 10],
    'reg_lambda': [0.1, 1, 5, 10, 20],
    'tree_method': ['hist'] 
    }

    xgb_random_search = RandomizedSearchCV(
        estimator=XGBRegressor(
            device="cuda",
            random_state=42,
            eval_metric='rmse'
        ),
        param_distributions=param_distributions,
        n_iter=50,
        cv=pds,
        scoring='neg_mean_squared_error',
        verbose=2,
        random_state=42,
        n_jobs=1
    )

    xgb_random_search.fit(X_search_g, y_search_g)

    best_params = xgb_random_search.best_params_
    print("\n--- Tuning Complete ---")
    print("Best Hyperparameters Found:", best_params)

    best_xgb_model = XGBRegressor(
        **best_params,
        device="cuda",
        random_state=42,
        eval_metric='rmse',
        early_stopping_rounds=15
    )

    best_xgb_model.fit(
        X_train_g,
        y_train_g,
        eval_set=[(X_val_g, y_val_g)],
        verbose=100
    )

    y_pred_gpu = best_xgb_model.predict(X_test_g)
    y_pred_cpu = cp.asnumpy(y_pred_gpu)

    mse_per_target = mean_squared_error(y_test, y_pred_cpu)
    r2_per_target = r2_score(y_test, y_pred_cpu)

    print(f"Target {col} -> MSE: {mse_per_target:.4f} | R² Score: {r2_per_target:.4f}")

In [74]:
for col in cols:   
    print(f"Current column: {col}")
    xgboost(X_other_pre, df[col], col)

Current column: score_z
Fitting 1 folds for each of 50 candidates, totalling 50 fits
[CV] END colsample_bytree=0.5, gamma=0.01, learning_rate=0.05, max_depth=5, min_child_weight=1, n_estimators=500, reg_alpha=1, reg_lambda=1, subsample=0.9, tree_method=hist; total time=   1.4s
[CV] END colsample_bytree=0.7, gamma=0.5, learning_rate=0.1, max_depth=6, min_child_weight=7, n_estimators=3000, reg_alpha=0.01, reg_lambda=1, subsample=0.6, tree_method=hist; total time=   4.2s
[CV] END colsample_bytree=0.5, gamma=0.01, learning_rate=0.1, max_depth=2, min_child_weight=3, n_estimators=500, reg_alpha=0.1, reg_lambda=1, subsample=0.8, tree_method=hist; total time=   0.6s
[CV] END colsample_bytree=0.9, gamma=0.5, learning_rate=0.1, max_depth=4, min_child_weight=7, n_estimators=500, reg_alpha=0.01, reg_lambda=5, subsample=1.0, tree_method=hist; total time=   0.5s
[CV] END colsample_bytree=0.5, gamma=0.5, learning_rate=0.02, max_depth=6, min_child_weight=10, n_estimators=2000, reg_alpha=0.1, reg_lambd

### Current Anime Prediction

In [2781]:
final_model.eval()

real_other_t = real_other.to(device)

with torch.no_grad():
    output = final_model(real_other_t)
    
    preds = output.cpu().numpy()



predictions = {}
for index, title in enumerate(real_df['title']):

    predictions[title] = [float(x) for x in preds[index]]

print(f"Collected predictions for {len(predictions)} items.")

sorted_predictions = dict(sorted(predictions.items(), key=lambda item: item[1][0], reverse=True))

from pprint import pprint
pprint(sorted_predictions, indent=4, sort_dicts=False)

Collected predictions for 72 items.
{   'Kusuriya no Hitorigoto 3rd Season': [   2.0891549587249756,
                                             1.2728078365325928,
                                             1.723383903503418,
                                             0.5110766291618347,
                                             1.04609215259552],
    'Ao Ashi Season 2': [   1.729020595550537,
                            1.0126206874847412,
                            1.2175542116165161,
                            0.3136860132217407,
                            0.8066071271896362],
    'Koori no Jouheki 2nd Season': [   1.6441385746002197,
                                       0.8017870187759399,
                                       1.066056728363037,
                                       0.0830940306186676,
                                       0.718880295753479],
    'Ao no Hako Season 2': [   1.4689122438430786,
                               1.2908415794372559,
     

In [2782]:
target_dir = "..\data\processed"
file_name = "overall_predictions.json" 
file_path = os.path.join(target_dir, file_name)

with open(file_path, "w", encoding="utf-8") as file:
    json.dump(sorted_predictions, file, indent=4)

<>:1: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:1: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
C:\Users\14793\AppData\Local\Temp\ipykernel_19312\1413168248.py:1: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
  target_dir = "..\data\processed"
